In [1]:
%cd ..

/Users/ya.pristalov/Documents/skating-project/ai-skating-assistant


In [26]:
from src.dataset import prepare_dataset
from src.config import VideoConfig, DataLoaderConfig
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import cv2
import torch

from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report,roc_auc_score
import joblib
import random

from tqdm.notebook import tqdm


In [3]:
SEED = 420

vc = VideoConfig(
    num_frames=32,
    target_fps=25,
    image_size=1024,
    return_meta=True,
)

# dc = DataLoaderConfig(
#     batch_size=1,
#     shuffle=True,
#     num_workers=1,
#     pin_memory=True,
#     persistent_workers=True,
#     prefetch_factor=2,
#     seed=SEED,
# )

dc = DataLoaderConfig(
    batch_size=4,
    shuffle=False,
    num_workers=1,
    pin_memory=False,
    persistent_workers=None,
    prefetch_factor=2,
    seed=SEED,
)

In [4]:
df, dataset, dataloader = prepare_dataset(exclude_videos=[1], video_config=vc, data_config=dc)

df

Videos in output dataset: [2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
Parsed 493 timecodes for 10 unique videos


,video_id,jump_id,jump_type,rotations,underrotation,edge,fall,t_start_val,t_end_val,duration
0,2,3,F,3,clean,NaN,0,00:10:43,00:10:45,3
1,2,4,Lz,3,ur,NaN,1,00:12:32,00:12:35,4
2,2,2,Lo,2,clean,NaN,0,00:13:01,00:13:03,3
3,2,5,A,2,clean,NaN,0,00:13:34,00:13:36,3
4,2,1,S,4,ur,NaN,1,00:18:13,00:18:15,3
...,...,...,...,...,...,...,...,...,...,...
488,11,2,Lo,3,clean,NaN,0,01:21:56,01:21:58,3
489,11,0,T,4,ur,NaN,1,01:26:13,01:26:15,3
490,11,1,S,3,clean,NaN,0,01:26:54,01:26:56,3
491,11,2,Lo,3,clean,NaN,0,01:27:06,01:27:08,3


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 493 entries, 0 to 492
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   video_id       493 non-null    int64  
 1   jump_id        493 non-null    int64  
 2   jump_type      493 non-null    str    
 3   rotations      493 non-null    int64  
 4   underrotation  493 non-null    str    
 5   edge           0 non-null      float64
 6   fall           493 non-null    object 
 7   t_start_val    493 non-null    object 
 8   t_end_val      493 non-null    object 
 9   duration       493 non-null    int64  
dtypes: float64(1), int64(4), object(3), str(2)
memory usage: 38.6+ KB


In [6]:
video_features = []
video_labels = []

frames_features: torch.Tensor 
label: torch.Tensor 

for el in tqdm(dataloader):
    frames_features, labels, metas = el
    labels = list(labels)
    print(f"{frames_features.shape = }")
    print(f"{labels = }")
    video_features.extend(frames_features.tolist())
    video_labels.extend(labels)
    

  0%|          | 0/124 [00:00<?, ?it/s]

idx = 0
idx = 1
idx = 2
idx = 3
idx = 4
frames_features.shape = torch.Size([4, 12])
labels = ['F', 'Lz', 'Lo', 'A']
idx = 5
idx = 6
idx = 7
idx = 8
frames_features.shape = torch.Size([4, 12])
labels = ['S', 'S', 'A', 'F']
idx = 9
idx = 10
idx = 11
idx = 12
frames_features.shape = torch.Size([4, 12])
labels = ['F', 'A', 'Lz', 'Lo']
idx = 13
idx = 14
idx = 15
idx = 16
frames_features.shape = torch.Size([4, 12])
labels = ['F', 'Lo', 'A', 'A']
idx = 17
idx = 18
idx = 19
idx = 20
frames_features.shape = torch.Size([4, 12])
labels = ['F', 'Lz', 'A', 'Lo']
idx = 21
idx = 22
idx = 23
idx = 24
frames_features.shape = torch.Size([4, 12])
labels = ['S', 'Lz', 'Lo', 'S']
idx = 25
idx = 26
idx = 27
idx = 28
frames_features.shape = torch.Size([4, 12])
labels = ['F', 'A', 'Lo', 'A']
idx = 29
idx = 30
idx = 31
idx = 32
frames_features.shape = torch.Size([4, 12])
labels = ['Lo', 'T', 'Lz', 'A']
idx = 33
idx = 34
idx = 35
idx = 36
frames_features.shape = torch.Size([4, 12])
labels = ['A', 'Lo', 'S', 'T'

In [7]:
len(video_features), len(video_features[0]), len(video_labels)

(493, 12, 493)

In [28]:
X = np.array(video_features)
y = np.array(video_labels)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y,
)

model = CatBoostClassifier(
    iterations=1000,
    depth=6,
    # learning_rate=0.03,
    loss_function="MultiClass",
    eval_metric="TotalF1",
    auto_class_weights="Balanced",
    random_seed=42,
    verbose=100,
)

model.fit(
    X_train,
    y_train,
    eval_set=(X_test, y_test),
    use_best_model=True,
)

y_pred = model.predict(X_test)
y_pred_random = random.choices(df["jump_type"].unique(), k=len(X_test))

print(classification_report(y_test, y_pred))
print(classification_report(y_test, y_pred_random))



Learning rate set to 0.105845
0:	learn: 0.3516761	test: 0.1122424	best: 0.1122424 (0)	total: 2.33ms	remaining: 2.33s
100:	learn: 0.8773363	test: 0.1162775	best: 0.1916063 (38)	total: 128ms	remaining: 1.14s
200:	learn: 0.9848662	test: 0.1244282	best: 0.1916063 (38)	total: 253ms	remaining: 1s
300:	learn: 0.9981273	test: 0.1445657	best: 0.1916063 (38)	total: 374ms	remaining: 867ms
400:	learn: 1.0000000	test: 0.1493954	best: 0.1916063 (38)	total: 495ms	remaining: 740ms
500:	learn: 1.0000000	test: 0.1428574	best: 0.1916063 (38)	total: 614ms	remaining: 612ms
600:	learn: 1.0000000	test: 0.1399089	best: 0.1916063 (38)	total: 742ms	remaining: 492ms
700:	learn: 1.0000000	test: 0.1480389	best: 0.1916063 (38)	total: 865ms	remaining: 369ms
800:	learn: 1.0000000	test: 0.1529330	best: 0.1916063 (38)	total: 993ms	remaining: 247ms
900:	learn: 1.0000000	test: 0.1462191	best: 0.1916063 (38)	total: 1.12s	remaining: 123ms
999:	learn: 1.0000000	test: 0.1462191	best: 0.1916063 (38)	total: 1.25s	remaining: 0u

In [9]:
import numpy as np

from catboost import CatBoostClassifier

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    HistGradientBoostingClassifier,
    StackingClassifier,
)
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression


X = np.array(video_features)
y = np.array(video_labels)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y,
)


catboost = CatBoostClassifier(
    iterations=3000,
    depth=6,
    learning_rate=0.03,
    loss_function="MultiClass",
    eval_metric="TotalF1",
    auto_class_weights="Balanced",
    random_seed=42,
    verbose=0,
    allow_writing_files=False,
)

extra_trees = ExtraTreesClassifier(
    n_estimators=1000,
    max_depth=None,
    min_samples_leaf=1,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)

random_forest = RandomForestClassifier(
    n_estimators=1000,
    max_depth=None,
    min_samples_leaf=1,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)

hist_gb = HistGradientBoostingClassifier(
    max_iter=1000,
    learning_rate=0.03,
    max_leaf_nodes=31,
    l2_regularization=0.1,
    random_state=42,
)

svc = make_pipeline(
    StandardScaler(),
    SVC(
        C=10,
        kernel="rbf",
        gamma="scale",
        probability=True,
        class_weight="balanced",
        random_state=42,
    ),
)


meta_model = LogisticRegression(
    max_iter=5000,
    class_weight="balanced",
    random_state=42,
)


ensemble = StackingClassifier(
    estimators=[
        ("catboost", catboost),
        ("extra_trees", extra_trees),
        ("random_forest", random_forest),
        ("hist_gb", hist_gb),
        ("svc", svc),
    ],
    final_estimator=meta_model,
    cv=StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42,
    ),
    stack_method="predict_proba",
    n_jobs=-1,
    passthrough=True,
)


ensemble.fit(X_train, y_train)

y_pred = ensemble.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           A       0.31      0.17      0.22        30
           F       0.29      0.29      0.29        21
          Lo       0.10      0.09      0.10        22
          Lz       0.13      0.12      0.12        17
           S       0.20      0.17      0.19        23
           T       0.16      0.45      0.23        11

    accuracy                           0.19       124
   macro avg       0.20      0.21      0.19       124
weighted avg       0.21      0.19      0.19       124

